# Project 4 — Your First Factor Study (Cross-Sectional Momentum)

Projects 2 and 3 looked at stocks one at a time. This is a different *kind* of question —
the kind real systematic funds actually ask:

> **Does a measurable characteristic of a stock predict whether it will beat the OTHER stocks next month?**

The characteristic we'll test is **momentum**: the tendency of stocks that have gone up a lot
recently to keep outperforming (and laggards to keep lagging). It's one of the most documented
patterns in all of finance — so this one has a real chance of "working."

**The method (rebalanced monthly):**
1. At the end of each month, rank all 10 stocks by their trailing 12-month return.
2. Form two baskets: the top 3 (**Winners**) and the bottom 3 (**Losers**).
3. Hold each basket through the *next* month, then re-rank and rebalance.
4. Compare Winners vs Losers vs just holding all 10 equally (the benchmark).

If momentum is real here, Winners should beat Losers. But hold your excitement — read the
honesty note at the end *before* you celebrate anything.

> Run top to bottom, Shift+Enter, kernel = **quant**.

## Step 1 — Monthly prices and returns

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

prices = pd.read_csv("../data/prices.csv", index_col=0, parse_dates=True)

# Momentum works on monthly horizons, so collapse daily prices to month-end.
# "ME" = month-end (pandas 3.0 spelling; older code used "M").
monthly_price = prices.resample("ME").last()
monthly_ret   = monthly_price.pct_change()

LOOKBACK = 12   # months of past return used to measure momentum
TOP_K    = 3    # how many stocks in each basket

print("Monthly data:", monthly_price.shape[0], "months x", monthly_price.shape[1], "stocks")
monthly_price.tail()

## Step 2 — The momentum signal (and the look-ahead guard, again)

The signal at the end of month *t* is each stock's return over the prior 12 months —
`price_now / price_12_months_ago - 1`. It uses only past prices, so it's knowable at time *t*.

Then the same critical idea as project 3: we **rank at the end of month t, but earn the returns
in month t+1**. `.shift(1)` on the basket membership enforces that — you can't hold a basket
you only chose *after* the month's returns happened. Forget it, and you get fake genius again.

In [ ]:
# Trailing LOOKBACK-month return, as of the end of each month
signal = monthly_price / monthly_price.shift(LOOKBACK) - 1

# Rank stocks each month: rank 1 = strongest momentum (ascending=False)
ranks = signal.rank(axis=1, ascending=False)

# Boolean baskets, decided at end of month t
winners_pick = ranks <= TOP_K                       # top 3
losers_pick  = ranks >= (ranks.max(axis=1).values.reshape(-1,1) - TOP_K + 1)
# (the line above marks the bottom 3; with 10 stocks that's ranks 8,9,10)

# Hold the baskets in the FOLLOWING month -> shift membership forward by 1
winners_held = winners_pick.shift(1)
losers_held  = losers_pick.shift(1)

winners_held.tail()

## Step 3 — Turn baskets into portfolio returns

In [ ]:
def basket_return(held_mask, rets):
    # equal-weight the stocks we're holding this month
    held = held_mask.astype(float)
    n_held = held.sum(axis=1)
    port = (rets * held).sum(axis=1) / n_held
    return port.replace([np.inf, -np.inf], np.nan)

winners = basket_return(winners_held, monthly_ret)
losers  = basket_return(losers_held,  monthly_ret)

# Benchmark: hold all 10 equally, every month (the "do nothing clever" baseline)
benchmark = monthly_ret.mean(axis=1)

# Academic long-short: long winners, short losers (illustration only — you can't easily short)
long_short = winners - losers

portfolios = pd.DataFrame({
    "Winners (top 3)": winners,
    "Losers (bottom 3)": losers,
    "Equal-weight all 10": benchmark,
    "Long-short (W - L)": long_short,
}).dropna()

portfolios.tail()

## Step 4 — Scorecard

In [ ]:
def metrics_monthly(ret):
    ret = ret.dropna()
    total  = (1 + ret).prod() - 1
    years  = len(ret) / 12
    cagr   = (1 + total) ** (1 / years) - 1
    vol    = ret.std() * np.sqrt(12)
    sharpe = (ret.mean() * 12) / vol if vol > 0 else np.nan
    eq     = (1 + ret).cumprod()
    maxdd  = (eq / eq.cummax() - 1).min()
    return pd.Series({"Total Return": total, "CAGR": cagr,
                      "Ann Vol": vol, "Sharpe": sharpe, "Max Drawdown": maxdd})

comp = pd.DataFrame({c: metrics_monthly(portfolios[c]) for c in portfolios.columns})

# display table built as text from scratch (keeps pandas 3.0 happy)
show = pd.DataFrame(index=comp.index, columns=comp.columns, dtype="object")
for row in ["Total Return", "CAGR", "Ann Vol", "Max Drawdown"]:
    show.loc[row] = (comp.loc[row] * 100).round(1).astype(str) + "%"
show.loc["Sharpe"] = comp.loc["Sharpe"].round(2).astype(str)
show

**Read it.** The key comparison is **Winners vs Losers**:
- If Winners beat Losers (higher return *and* higher Sharpe), momentum showed up in your data.
- Also check Winners vs **Equal-weight all 10** — beating the simple benchmark is the bar that
  actually matters. Beating the losers but not the benchmark isn't worth much.

## Step 5 — The chart

In [ ]:
equity = (1 + portfolios).cumprod()

plt.figure(figsize=(12, 6))
for col in ["Winners (top 3)", "Losers (bottom 3)", "Equal-weight all 10"]:
    plt.plot(equity.index, equity[col], label=col, lw=1.5)
plt.title("Growth of $1 — momentum baskets vs the benchmark")
plt.ylabel("Value ($, started at 1)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---
## The honesty note — read this BEFORE you get excited

Say Winners beat Losers and even beat the benchmark. Tempting to conclude "momentum works,
I've found an edge!" **Don't.** Here's why a real quant stays sceptical of this exact result:

1. **Tiny universe.** You ranked *10 stocks* and held *3*. Picking 3 names from 10 is almost
   pure noise — one lucky winner (hello, FMG) can carry the whole "Winners" basket and make
   momentum look brilliant when it was really just one stock. Real momentum studies use
   hundreds or thousands of stocks for a reason.

2. **One market, one decade.** Australian large-caps, ~2015–2026. Whatever you found might be
   specific to this slice and vanish elsewhere.

3. **In-sample.** You measured momentum on the *same data* you're judging it on. You haven't
   yet checked whether it would have worked on data the strategy never saw — which is the only
   test that counts.

4. **No costs.** Monthly rebalancing means trading every month. Add realistic costs and a thin
   edge can disappear entirely (you saw exactly this in project 3).

So the correct reaction to a "winning" factor isn't excitement — it's: *"interesting signal,
now let me try to destroy it."*

### What's next
That destruction test is **Project 5: the out-of-sample / train-test split** — the most
important validation step in the whole roadmap. We take whatever you found here, hide half the
data, build the rule on the half we can see, and check whether it still holds on the half it's
never seen. If the edge survives that, it's worth a second look. If it evaporates — and it
often does — you just saved yourself from betting real money on a mirage.

### Try first
- Change `TOP_K = 2` or `4`, or `LOOKBACK = 6` or `3`, and re-run. Does the "edge" hold up, or
  swing around wildly? If it swings around, that's your tiny-universe noise talking.